# 12. Containerization with Docker

In the previous notebook, we served the trained model through a local FastAPI service and connected a Streamlit frontend to it. This is already close to a real application setup, but it still depends heavily on the local Python environment.

Docker helps us package the application environment in a reproducible way. Instead of asking every user to install the exact same Python packages manually, we define the runtime once in a Dockerfile and run the application inside a container.

## 1. Why Containerization Matters

A machine learning application usually needs more than a model file. It also needs preprocessing code, serving code, Python packages, system dependencies, configuration, and a clear way to start the service.

Without containers, the application can work on one laptop and fail on another because package versions, Python versions, or paths are different. With Docker, we make the runtime explicit.

For this project, containerization is useful because we have two services:

- a FastAPI service that exposes the model as an API
- a Streamlit app that provides a simple user interface

These services should run together, but they have different responsibilities.

## 2. Image vs. Container

A Docker image is the packaged application environment. It contains the operating system layer, Python, installed packages, and application code.

A Docker container is a running instance of an image.

A useful mental model is:

- `Dockerfile` defines how to build the image
- `docker build` creates the image
- `docker run` starts one container from that image
- `docker compose` starts multiple connected containers together

## 3. API Container

The FastAPI service is defined in `src/serving/api.py`. The API Dockerfile installs the serving dependencies, copies the source code, and starts Uvicorn.

Relevant file: `Dockerfile.api`

The container starts this command:

```bash
uvicorn src.serving.api:app --host 0.0.0.0 --port 8000
```

`0.0.0.0` is important inside a container. It means the service listens on all network interfaces in the container, so Docker can forward the port to the host machine.

## 4. Streamlit Container

The Streamlit app is defined in `app/streamlit_app.py`. It does not load the model directly. Instead, it sends the customer data to the FastAPI service and displays the prediction response.

Relevant file: `Dockerfile.streamlit`

The Dockerfiles do not use the full development `requirements.txt`. The API image uses `requirements-api.txt`, because it must load the preprocessing and XGBoost model artifacts. The Streamlit image uses `requirements-streamlit.txt`, because it only needs the frontend package and an HTTP client.

This keeps both images focused on what they actually need at inference time and avoids installing notebook, training, and optional deep learning packages.

In `Dockerfile.api`, `xgboost` is installed separately with `--no-deps`. For this CPU-based demo, we do not need optional GPU-related packages that recent XGBoost wheels may otherwise pull into the image.

The API image installs dependencies before copying `src/`. This improves Docker layer caching: changes in application code do not force a full dependency reinstall.

The container starts this command:

```bash
streamlit run app/streamlit_app.py --server.address=0.0.0.0 --server.port=8501
```

The API URL is not hard-coded for production use. The app reads it from the environment variable `API_URL`. In Docker Compose, this value is set to `http://api:8000`, because `api` is the service name inside the Compose network.

## 5. Docker Compose

Docker Compose lets us start both services together. The file `docker-compose.yml` defines two services:

- `api`: builds `Dockerfile.api` and exposes port `8000`
- `streamlit`: builds `Dockerfile.streamlit` and exposes port `8501`

The Streamlit service depends on the API service and uses the internal URL `http://api:8000`.

The API service mounts the local `models/` directory into the container:

```yaml
volumes:
  - ./models:/app/models:ro
  - ./monitoring:/app/monitoring
```

`ro` means read-only. The serving container can read the model artifacts, but it should not modify them.

The `monitoring/` mount is writable. Prediction logs are operational output of the running service, so they need a persistent location outside the disposable container filesystem.

## 6. Why the Model Is Not Built into the Image

In this project, the model artifact is produced by the training pipeline and stored locally in `models/`. The API container reads the latest model path from `models/model_manifest.json`.

We do not copy the model into the image here because code and model artifacts often have different lifecycles:

- serving code changes when the application changes
- model artifacts change when a model is retrained
- a team may want to promote a new model without rebuilding all application code

For a local teaching setup, mounting `models/` is simple and transparent. In a cloud environment, the model might instead come from an artifact store, a model registry, or object storage.

## 7. Build and Start the Application

Run the following commands from the repository root.

Build the images:

```bash
docker compose build
```

Start both services:

```bash
docker compose up
```

Open the FastAPI Swagger UI:

```text
http://127.0.0.1:8000/docs
```

Open the Streamlit app:

```text
http://127.0.0.1:8501
```

Stop the services:

```bash
docker compose down
```

## 8. Local CI/CD Perspective

This setup is also useful for the next step: GitHub Actions.

A CI workflow can check whether the application still builds successfully:

```bash
docker compose build
```

It can also start the API container and call a health endpoint:

```bash
curl http://127.0.0.1:8000/health
```

This is not yet a full deployment pipeline. It is a compact demonstration of a common CI/CD idea: before code is merged or released, we automatically verify that the application can be built and started in a clean environment.

## 9. Reflection

The local notebook workflow was useful for exploration, training, and learning. The refactored pipeline made the process executable. The API made the model available to another application. Docker now packages the runtime so the application can be started more consistently.

This is an important step from model development toward an ML application. The trained model is no longer just a file on disk. It becomes part of a small system with clear interfaces:

- training produces artifacts
- the API loads artifacts and serves predictions
- the frontend consumes the API
- Docker Compose starts the full local application stack

In production, additional topics would become important: image registries, secrets management, environment-specific configuration, deployment targets, monitoring, and rollback strategies.